# NGIML Inference

In [ ]:
import subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/juhenes/ngiml"
REPO_BRANCH = "main"
REPO_DIR = Path("/content/ngiml")

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "origin", REPO_BRANCH], check=True)
else:
    subprocess.run([
        "git",
        "clone",
        "--branch",
        REPO_BRANCH,
        "--single-branch",
        REPO_URL,
        str(REPO_DIR),
    ], check=True)

sys.path.insert(0, str(REPO_DIR))
print(f"Repo ready at {REPO_DIR} on branch {REPO_BRANCH}")

In [ ]:
from __future__ import annotations

from pathlib import Path

import torch

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

CHECKPOINT_PATH = Path('/content/drive/MyDrive/thesis/ngiml/checkpoints/best_checkpoint.pt')
CHECKPOINT_DIR = CHECKPOINT_PATH.parent
HF_DATASET_REPO_ID = 'juhenes/ngiml-test'
DRIVE_OUTPUT_ROOT = Path('/content/drive/MyDrive/thesis/ngiml-casia-inference')
HF_SNAPSHOT_LOCAL_DIR = Path('/content/hf_datasets/ngiml_test')

INFERENCE_STRATEGY = 'direct'
INFERENCE_BATCH_SIZE = 64
SWEEP_SAMPLE_INDEX = 0
SWEEP_DATASET_NAME = None
SWEEP_PLOT_MAX_CHECKPOINTS = 8
THRESHOLD_FOR_METRICS = None
PLOT_BINARY_THRESHOLD = 0.5

CSV_OUTPUT_DIR = DRIVE_OUTPUT_ROOT / 'csv'
PLOT_OUTPUT_DIR = DRIVE_OUTPUT_ROOT / 'plots'
CSV_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for p in [start, *start.parents, Path('/content/ngiml')]:
        if (p / 'tools' / 'infer_helpers.py').exists() and (p / 'src').exists():
            return p
    raise FileNotFoundError('Could not find NGIML repo root.')

REPO_ROOT = find_repo_root()
import sys
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from tools.infer_helpers import run_prepared_test_inference_from_hf_dataset, sweep_checkpoint_inference_for_prepared_sample_from_hf_dataset

assert CHECKPOINT_PATH.exists(), f'Checkpoint not found: {CHECKPOINT_PATH}'

In [ ]:
run = run_prepared_test_inference_from_hf_dataset(
    checkpoint_path=CHECKPOINT_PATH,
    hf_dataset_repo_id=HF_DATASET_REPO_ID,
    snapshot_local_dir=HF_SNAPSHOT_LOCAL_DIR,
    output_root=DRIVE_OUTPUT_ROOT,
    inference_strategy=INFERENCE_STRATEGY,
    inference_batch_size=INFERENCE_BATCH_SIZE,
    threshold_for_metrics=THRESHOLD_FOR_METRICS,
    plot_binary_threshold=PLOT_BINARY_THRESHOLD,
)

summary_df = run['summary_df']
results_df = run['results_df']
print('Snapshot:', run['snapshot_path'])
print('Device:', run['device'])
print('Normalization:', run['normalization_mode'])
print('Inference batch size:', run['inference_batch_size'])
print('Threshold for CSV metrics:', run['threshold_used'])
print('Plot threshold:', run['plot_binary_threshold'])
print('Saved full CSV:', run['results_csv'])
print('Saved summary CSV:', run['summary_csv'])
print('Saved plot root:', run['plot_output_dir'])
display(summary_df)


In [ ]:
import pandas as pd

sweep_run = sweep_checkpoint_inference_for_prepared_sample_from_hf_dataset(
    checkpoint_dir=CHECKPOINT_DIR,
    hf_dataset_repo_id=HF_DATASET_REPO_ID,
    snapshot_local_dir=HF_SNAPSHOT_LOCAL_DIR,
    normalization_mode=run['normalization_mode'],
    strategy=INFERENCE_STRATEGY,
    threshold=THRESHOLD_FOR_METRICS,
    sample_index=SWEEP_SAMPLE_INDEX,
    dataset_name=SWEEP_DATASET_NAME,
    plot_max_checkpoints=SWEEP_PLOT_MAX_CHECKPOINTS,
)

selected = sweep_run['selected_sample']
sweep_df = pd.DataFrame(sweep_run['records'])
print('Sweep checkpoint dir:', CHECKPOINT_DIR)
print('Sweep snapshot:', sweep_run['snapshot_path'])
print('Sweep sample URI:', selected['sample_uri'])
print('Sweep sample dataset:', selected['dataset'])
print('Sweep sample label:', selected['label'])
display(sweep_df)


In [ ]:
import subprocess
import sys
from pathlib import Path

import torch

from tools.infer_helpers import load_model_from_checkpoint

try:
    from thop import clever_format, profile
except Exception:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "thop"])
    from thop import clever_format, profile

checkpoint_dict_content = torch.load(CHECKPOINT_PATH, map_location="cpu")
training_config = checkpoint_dict_content.get("training_config", {})

model, _, ckpt_info = load_model_from_checkpoint(CHECKPOINT_PATH)
model = model.cpu().eval()

input_size = int(training_config.get("input_size", 448))

class _NgimlForProfiling(torch.nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.base_model = base_model

    def forward(self, x):
        out = self.base_model(x, target_size=x.shape[-2:], residual_noise=None)
        if isinstance(out, (list, tuple)):
            return out[0]
        return out

wrapper = _NgimlForProfiling(model).eval()
dummy = torch.randn(1, 3, input_size, input_size, dtype=torch.float32)

with torch.no_grad():
    macs, params = profile(wrapper, inputs=(dummy,), verbose=False)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
flops = 2.0 * macs

macs_hr, params_hr = clever_format([macs, params], "%.3f")
flops_hr = clever_format([flops], "%.3f")

print("Checkpoint:", CHECKPOINT_PATH)
print("Input shape:", tuple(dummy.shape))
print("Trainable params:", f"{trainable_params:,}")
print("Total params:", f"{total_params:,}")
print("THOP params:", params_hr)
print("MACs:", macs_hr)
print("Approx FLOPs (2 * MACs):", flops_hr)